# stg_traffic_v3 — Silver Cleaning for NYC Traffic Volume Counts
## Medallion Architecture: Bronze → Silver

**MSBA 305 — Google Colab Version**

**What this notebook does:**
Reads `bronze_traffic` (Delta table from Drive), applies all 9 cleaning steps, aggregates 15-min → hourly per borough, writes to
`silver_stg_traffic`.

### Changes from v2
| Change | Reason |
|--------|--------|
| All column names lowercased | Bronze ingestion lowercases everything |
| Read via Delta path on Drive | Colab version — no managed tables |
| Write via `.save()` to Drive path | Colab version — replaces `saveAsTable()` |
| Deduplication added before Step 1 | Bronze uses `mode("append")` |
| Step 7 rewritten | Source data already has full borough names, not abbreviations |
| Borough format: title case | `Manhattan`, `Brooklyn`, etc. — matches source data |

### Steps
| Step | What it does | Status |
|------|-------------|--------|
| 1 | Validate yr, m, d, hh | Done |
| 2 | Reconstruct unified timestamp | Done (mm included) |
| 3 | Reproject wktgeom EPSG:2263 → 4326 | Done (graceful skip if no pyproj) |
| 4 | Handle fromst / tost missing values | Done — Dictionary recovery |
| 5a | Validate requestid + vol | Done |
| 5b | Recover segmentid + street | Done — Dictionary recovery |
| 6 | Verify vol=0 preserved | Done |
| 7 | Standardize borough names | Done (validate, not remap) |
| 8 | Validate borough — spatial recovery + 5% threshold | Done |
| 9 | Aggregate 15-min → hourly per borough | Done |


In [62]:
import datetime, uuid, json
from functools import reduce
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from notebookutils import mssparkutils
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Runtime identifiers
today = datetime.date.today()
now = datetime.datetime.utcnow()
ingest_run_id = str(uuid.uuid4())

print("Spark ready")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 64, Finished, Available, Finished, False)

Spark ready


In [63]:
# Table names for Fabric Lakehouse
BRONZE_TABLE = "bronze_traffic_nyc"
SILVER_TABLE = "silver_stg_traffic"
QUARANTINE_TABLE = "quarantine_stg_traffic"

print(f"Bronze source: {BRONZE_TABLE}")
print(f"Silver target: {SILVER_TABLE}")
print(f"Quarantine table: {QUARANTINE_TABLE}")


# Check for spatial packages
SPATIAL_AVAILABLE = False
try:
    import pyproj
    from shapely.wkt import loads as wkt_loads
    from shapely.geometry import Point, shape
    SPATIAL_AVAILABLE = True
    print("✓ pyproj and shapely available — spatial features enabled")
except ImportError as e:
    print(f"Spatial packages not available: {e}")
    print("  Steps 3 and 8 spatial features will be skipped.")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 65, Finished, Available, Finished, False)

Bronze source: bronze_traffic_nyc
Silver target: silver_stg_traffic
Quarantine table: quarantine_stg_traffic
Spatial packages not available: No module named 'pyproj'
  Steps 3 and 8 spatial features will be skipped.


## Read Bronze + Deduplicate

In [64]:
# ── ONE-TIME PURGE — run once to test incremental write, then skip ────────────
'''
for tbl in [SILVER_TABLE, QUARANTINE_TABLE]:
    try:
        spark.sql(f"TRUNCATE TABLE {tbl}")
        print(f" Truncated: {tbl}")
    except Exception as e:
        print(f"  {tbl} doesn't exist yet, skipping: {e}")

print("\nPurge complete — re-run pipeline from the top.")
'''

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 66, Finished, Available, Finished, False)

'\nfor tbl in [SILVER_TABLE, QUARANTINE_TABLE]:\n    try:\n        spark.sql(f"TRUNCATE TABLE {tbl}")\n        print(f"✅ Truncated: {tbl}")\n    except Exception as e:\n        print(f"⚠️  {tbl} doesn\'t exist yet, skipping: {e}")\n\nprint("\nPurge complete — re-run pipeline from the top.")\n'

In [65]:
df = spark.table(BRONZE_TABLE)

initial_count = df.count()
print(f"Bronze traffic loaded: {initial_count:,} rows")
print(f"   Columns ({len(df.columns)}): {df.columns}")
df.printSchema()

assert initial_count > 0, "PIPELINE ERROR: Bronze traffic table is empty!"

# Rename to lowercase for consistency
df = df.select([F.col(c).alias(c.lower()) for c in df.columns])

# Check for wktgeom
HAS_WKTGEOM = "wktgeom" in df.columns
if HAS_WKTGEOM:
    print("\nwktgeom column found — spatial features enabled")
else:
    print("\nwktgeom NOT found — spatial features disabled")

# Create timestamp from date components if it doesn't exist
if "timestamp" not in df.columns:
    df = df.withColumn(
        "timestamp",
        F.to_timestamp(F.concat_ws("-", F.col("yr"), F.col("m"), F.col("d"), F.col("hh"), F.col("mm")), "yyyy-M-d-H-m")
    )

# Now your original dedup code works
pre_dedup = df.count()
if "ingested_at" in df.columns:
    w_dedup = Window.partitionBy("requestid", "segmentid", "timestamp").orderBy(F.desc("ingested_at"))
    df = (df.withColumn("_rn", F.row_number().over(w_dedup))
          .filter(F.col("_rn") == 1)
          .drop("_rn"))
else:
    df = df.dropDuplicates(["requestid", "segmentid", "timestamp"])

post_dedup = df.count()
print(f"\nDeduplication: {pre_dedup:,} -> {post_dedup:,} ({pre_dedup - post_dedup:,} duplicates removed)")

# Filter to 2026 only
pre_year = df.count()
print("\nYear distribution:")
df.groupBy("yr").count().orderBy("yr").show(truncate=False)
df = df.filter(F.col("yr") == 2026)
post_year = df.count()
print(f"Filtered to 2026: {pre_year:,} -> {post_year:,}")

row_log = [("Bronze (after dedup + 2026 filter)", post_year)]

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 67, Finished, Available, Finished, False)

Bronze traffic loaded: 9,120 rows
   Columns (18): ['RequestID', 'Boro', 'Yr', 'M', 'D', 'HH', 'MM', 'Vol', 'SegmentID', 'WktGeom', 'street', 'fromSt', 'toSt', 'Direction', 'source', 'ingest_run_id', 'ingested_at', 'ingest_date']
root
 |-- RequestID: integer (nullable = true)
 |-- Boro: string (nullable = true)
 |-- Yr: integer (nullable = true)
 |-- M: integer (nullable = true)
 |-- D: integer (nullable = true)
 |-- HH: integer (nullable = true)
 |-- MM: integer (nullable = true)
 |-- Vol: integer (nullable = true)
 |-- SegmentID: integer (nullable = true)
 |-- WktGeom: string (nullable = true)
 |-- street: string (nullable = true)
 |-- fromSt: string (nullable = true)
 |-- toSt: string (nullable = true)
 |-- Direction: string (nullable = true)
 |-- source: string (nullable = true)
 |-- ingest_run_id: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- ingest_date: string (nullable = true)


wktgeom column found — spatial features enabled

Deduplication: 9,120 

## Step 1: Validate Date and Time Columns

In [66]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: Validate date/time columns
# ═══════════════════════════════════════════════════════════════════════════════

before = df.count()

for col_name in ["yr", "m", "d", "hh"]:
    null_count = df.filter(F.col(col_name).isNull()).count()
    if null_count > 0:
        print(f"   Dropping {null_count} rows with NULL {col_name}")
        df = df.filter(F.col(col_name).isNotNull())

bad_month = df.filter((F.col("m") < 1) | (F.col("m") > 12)).count()
if bad_month > 0:
    print(f"   Dropping {bad_month} rows with m outside [1, 12]")
    df = df.filter((F.col("m") >= 1) & (F.col("m") <= 12))

bad_day = df.filter((F.col("d") < 1) | (F.col("d") > 31)).count()
if bad_day > 0:
    print(f"   Dropping {bad_day} rows with d outside [1, 31]")
    df = df.filter((F.col("d") >= 1) & (F.col("d") <= 31))

bad_hour = df.filter((F.col("hh") < 0) | (F.col("hh") > 23)).count()
if bad_hour > 0:
    print(f"   Dropping {bad_hour} rows with hh outside [0, 23]")
    df = df.filter((F.col("hh") >= 0) & (F.col("hh") <= 23))

# Flag invalid dates
df = df.withColumn(
    "_flag_bad_date",
    F.when(
        (F.col("yr").isNull()) | 
        (F.col("m").isNull()) | 
        (F.col("d").isNull()) | 
        (F.col("hh").isNull()),
        "NULL_DATE_COMPONENTS"
    ).otherwise(None)
)

after = df.count()
print(f"\nStep 1 complete: {before - after:,} rows dropped, {after:,} remaining")
row_log.append(("After Step 1 (date validation)", after))

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 68, Finished, Available, Finished, False)


Step 1 complete: 0 rows dropped, 9,120 remaining


## Step 2: Reconstruct Unified Timestamp

In [67]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: Reconstruct unified timestamp
# ═══════════════════════════════════════════════════════════════════════════════

if "timestamp" in df.columns:
    df = df.drop("timestamp")

df = df.withColumn(
    "timestamp",
    F.to_timestamp(
        F.concat_ws(
            "-",
            F.col("yr").cast("string"),
            F.lpad(F.col("m").cast("string"), 2, "0"),
            F.lpad(F.col("d").cast("string"), 2, "0"),
            F.lpad(F.col("hh").cast("string"), 2, "0"),
            F.lpad(F.coalesce(F.col("mm"), F.lit(0)).cast("string"), 2, "0"),
        ),
        "yyyy-MM-dd-HH-mm"
    )
)

null_ts = df.filter(F.col("timestamp").isNull()).count()
if null_ts > 0:
    print(f"{null_ts} rows produced NULL timestamp (invalid date combos like Feb 30)")
    df.filter(F.col("timestamp").isNull()).select("yr", "m", "d", "hh", "mm").show(10)
    df = df.filter(F.col("timestamp").isNotNull())

after = df.count()
time_range = df.agg(F.min("timestamp").alias("earliest"), F.max("timestamp").alias("latest")).collect()[0]
print(f"Step 2 complete: timestamp reconstructed")
print(f"   Time range: {time_range['earliest']} -> {time_range['latest']}")
print(f"   Rows: {after:,}")
row_log.append(("After Step 2 (timestamp rebuild)", after))

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 69, Finished, Available, Finished, False)

Step 2 complete: timestamp reconstructed
   Time range: 2026-01-03 00:00:00 -> 2026-02-03 23:45:00
   Rows: 9,120


## Step 3: Reproject wktgeom (EPSG:2263 → EPSG:4326)

In [68]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: Reproject wktgeom from EPSG:2263 to EPSG:4326
# ═══════════════════════════════════════════════════════════════════════════════

REPROJECTION_DONE = False

if not HAS_WKTGEOM:
    print("Step 3 skipped: wktgeom not in Bronze table")
elif not SPATIAL_AVAILABLE:
    print("Step 3 skipped: pyproj/shapely not installed")
else:
    from pyproj import Transformer
    from shapely.wkt import loads as wkt_loads

    transformer = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True)

    @F.udf(DoubleType())
    def extract_lat(wkt_str):
        if wkt_str is None:
            return None
        try:
            geom = wkt_loads(wkt_str)
            lon, lat = transformer.transform(geom.x, geom.y)
            return lat
        except Exception:
            return None

    @F.udf(DoubleType())
    def extract_lon(wkt_str):
        if wkt_str is None:
            return None
        try:
            geom = wkt_loads(wkt_str)
            lon, lat = transformer.transform(geom.x, geom.y)
            return lon
        except Exception:
            return None

    print("Reprojecting wktgeom coordinates...")
    df = df.withColumn("sensor_lat", extract_lat(F.col("wktgeom")))
    df = df.withColumn("sensor_lon", extract_lon(F.col("wktgeom")))

    valid_coords = df.filter(F.col("sensor_lat").isNotNull() & F.col("sensor_lon").isNotNull()).count()
    null_coords = df.filter(F.col("sensor_lat").isNull() | F.col("sensor_lon").isNull()).count()

    print(f"Step 3 complete: coordinates reprojected")
    print(f"   Valid: {valid_coords:,}, NULL: {null_coords:,}")
    df.filter(F.col("sensor_lat").isNotNull()).select("wktgeom", "sensor_lat", "sensor_lon").show(3, truncate=False)
    REPROJECTION_DONE = True

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 70, Finished, Available, Finished, False)

Step 3 skipped: pyproj/shapely not installed


## Step 4: Handle fromst and tost Missing Values

In [69]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: Handle fromst and tost missing values
# ═══════════════════════════════════════════════════════════════════════════════

def build_recovery_dict(df, key_col, value_col):
    pairs = (
        df.filter(F.col(key_col).isNotNull() & F.col(value_col).isNotNull())
          .select(key_col, value_col).distinct().collect()
    )
    return {row[key_col]: row[value_col] for row in pairs}

def recover_column(df, target_col, key_col, dictionary, label):
    if not dictionary:
        print(f"     {label}: dictionary empty — no recovery possible")
        return df, 0
    null_before = df.filter(F.col(target_col).isNull()).count()
    if null_before == 0:
        print(f"     {label}: no NULLs to recover")
        return df, 0
    mapping = F.create_map(*[item for k, v in dictionary.items() for item in (F.lit(k), F.lit(v))])
    df = df.withColumn(
        target_col,
        F.when(F.col(target_col).isNull(), mapping[F.col(key_col)]).otherwise(F.col(target_col))
    )
    null_after = df.filter(F.col(target_col).isNull()).count()
    recovered = null_before - null_after
    print(f"     {label}: recovered {recovered:,} / {null_before:,} NULLs")
    return df, recovered

# 4a: Normalize sentinel values to NULL
print("Normalizing sentinel values...")
if "fromst" in df.columns:
    bad_from = df.filter(F.col("fromst") == "0.000 Undefined").count()
    df = df.withColumn("fromst", F.when(F.col("fromst") == "0.000 Undefined", None).otherwise(F.col("fromst")))
    print(f"   fromst: {bad_from:,} '0.000 Undefined' -> NULL")

if "tost" in df.columns:
    bad_to = df.filter((F.col("tost") == "") | (F.col("tost").isNull())).count()
    df = df.withColumn("tost", F.when(F.col("tost") == "", None).otherwise(F.col("tost")))
    print(f"   tost: {bad_to:,} blank/NULL values")

# 4b-c: Dictionary recovery for fromst
if "fromst" in df.columns:
    print("\nRecovering fromst...")
    if "segmentid" in df.columns:
        seg_from = build_recovery_dict(df, "segmentid", "fromst")
        df, _ = recover_column(df, "fromst", "segmentid", seg_from, "Level 1 (segmentid)")
    if HAS_WKTGEOM:
        wkt_from = build_recovery_dict(df, "wktgeom", "fromst")
        df, _ = recover_column(df, "fromst", "wktgeom", wkt_from, "Level 2 (wktgeom)")
    remaining = df.filter(F.col("fromst").isNull()).count()
    print(f"   Final fromst NULLs: {remaining:,}")

# 4d: Dictionary recovery for tost
if "tost" in df.columns:
    print("\nRecovering tost...")
    if "segmentid" in df.columns:
        seg_to = build_recovery_dict(df, "segmentid", "tost")
        df, _ = recover_column(df, "tost", "segmentid", seg_to, "Level 1 (segmentid)")
    if HAS_WKTGEOM:
        wkt_to = build_recovery_dict(df, "wktgeom", "tost")
        df, _ = recover_column(df, "tost", "wktgeom", wkt_to, "Level 2 (wktgeom)")
    remaining = df.filter(F.col("tost").isNull()).count()
    print(f"   Final tost NULLs: {remaining:,}")

print(f"\nStep 4 complete")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 71, Finished, Available, Finished, False)

Normalizing sentinel values...
   fromst: 0 '0.000 Undefined' -> NULL
   tost: 0 blank/NULL values

Recovering fromst...
     Level 1 (segmentid): no NULLs to recover
     Level 2 (wktgeom): no NULLs to recover
   Final fromst NULLs: 0

Recovering tost...
     Level 1 (segmentid): no NULLs to recover
     Level 2 (wktgeom): no NULLs to recover
   Final tost NULLs: 0

Step 4 complete


## Step 5a: Validate requestid and vol

In [70]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5a: Validate requestid and vol
# ═══════════════════════════════════════════════════════════════════════════════

before = df.count()

null_rid = df.filter(F.col("requestid").isNull()).count()
if null_rid > 0:
    print(f"   Dropping {null_rid} rows with NULL requestid")
    df = df.filter(F.col("requestid").isNotNull())

null_vol = df.filter(F.col("vol").isNull()).count()
if null_vol > 0:
    print(f"   Dropping {null_vol} rows with NULL vol")
    df = df.filter(F.col("vol").isNotNull())

neg_vol = df.filter(F.col("vol") < 0).count()
if neg_vol > 0:
    print(f"   Dropping {neg_vol} rows with negative vol")
    df = df.filter(F.col("vol") >= 0)
    
# Flag invalid requestid and volume
df = df.withColumn(
    "_flag_bad_requestid_vol",
    F.when(
        (F.col("requestid").isNull()) | 
        (F.col("vol").isNull()) |
        (F.col("vol") < 0),
        "BAD_REQUESTID_OR_VOLUME"
    ).otherwise(None)
)

after = df.count()
print(f"\nStep 5a complete: {before - after:,} dropped, {after:,} remaining")
row_log.append(("After Step 5a (requestid + vol)", after))

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 72, Finished, Available, Finished, False)


Step 5a complete: 0 dropped, 9,120 remaining


## Step 5b: Recover segmentid and street

In [71]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5b: Recover segmentid and street via dictionary lookup
# ═══════════════════════════════════════════════════════════════════════════════

if "segmentid" in df.columns:
    print("Recovering segmentid...")
    null_before = df.filter(F.col("segmentid").isNull()).count()
    if null_before > 0 and HAS_WKTGEOM:
        wkt_seg = build_recovery_dict(df, "wktgeom", "segmentid")
        df, _ = recover_column(df, "segmentid", "wktgeom", wkt_seg, "Level 1 (wktgeom)")
    null_after = df.filter(F.col("segmentid").isNull()).count()
    print(f"   segmentid NULLs: {null_before:,} -> {null_after:,}")

if "street" in df.columns:
    print("\nRecovering street...")
    null_before = df.filter(F.col("street").isNull()).count()
    if null_before > 0:
        if HAS_WKTGEOM:
            wkt_st = build_recovery_dict(df, "wktgeom", "street")
            df, _ = recover_column(df, "street", "wktgeom", wkt_st, "Level 1 (wktgeom)")
        if "segmentid" in df.columns:
            seg_st = build_recovery_dict(df, "segmentid", "street")
            df, _ = recover_column(df, "street", "segmentid", seg_st, "Level 2 (segmentid)")
    null_after = df.filter(F.col("street").isNull()).count()
    print(f"   street NULLs: {null_before:,} -> {null_after:,}")

print(f"\nStep 5b complete")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 73, Finished, Available, Finished, False)

Recovering segmentid...
   segmentid NULLs: 0 -> 0

Recovering street...
   street NULLs: 0 -> 0

Step 5b complete


## Step 6: Verify vol = 0 Is Preserved

In [72]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: Verify vol = 0 rows are preserved
# ═══════════════════════════════════════════════════════════════════════════════

zero_vol = df.filter(F.col("vol") == 0).count()
total = df.count()
pct = zero_vol / total * 100 if total > 0 else 0

print(f"Step 6: vol = 0 rows preserved")
print(f"   Zero-volume rows: {zero_vol:,} ({pct:.2f}%)")
if zero_vol == 0:
    print("   WARNING: No vol = 0 rows — unusual, verify data")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 74, Finished, Available, Finished, False)

Step 6: vol = 0 rows preserved
   Zero-volume rows: 41 (0.45%)


## Step 7: Standardize Borough Names

In [73]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: Standardize borough names
# ═══════════════════════════════════════════════════════════════════════════════

VALID_BOROUGHS = ["Manhattan", "Brooklyn", "Bronx", "Queens", "Staten Island"]

print("Current boro values:")
df.groupBy("boro").count().orderBy(F.desc("count")).show(truncate=False)

df = df.withColumn("boro_clean", F.initcap(F.trim(F.col("boro"))))

borough_expr = (
    F.when(F.col("boro_clean") == "Manhattan", "Manhattan")
     .when(F.col("boro_clean") == "Brooklyn", "Brooklyn")
     .when(F.col("boro_clean") == "Bronx", "Bronx")
     .when(F.col("boro_clean") == "Queens", "Queens")
     .when(F.col("boro_clean") == "Staten Island", "Staten Island")
     .otherwise(None)
)
df = df.withColumn("borough", borough_expr).drop("boro_clean")

unmapped = df.filter(F.col("borough").isNull() & F.col("boro").isNotNull()).count()
if unmapped > 0:
    print(f"\n{unmapped:,} rows with unrecognised boro values (set to NULL):")
    df.filter(F.col("borough").isNull() & F.col("boro").isNotNull())       .groupBy("boro").count().show(truncate=False)

# Flag rows with NULL borough after standardization
df = df.withColumn(
    "_flag_bad_borough",
    F.when(F.col("borough").isNull(), "NULL_BOROUGH").otherwise(None)
)

print(f"\nStep 7 complete")
print("Borough distribution after standardization:")
df.groupBy("borough").count().orderBy("borough").show(truncate=False)

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 75, Finished, Available, Finished, False)

Current boro values:
+---------+-----+
|boro     |count|
+---------+-----+
|Bronx    |5376 |
|Brooklyn |2688 |
|Manhattan|672  |
|Queens   |384  |
+---------+-----+


Step 7 complete
Borough distribution after standardization:
+---------+-----+
|borough  |count|
+---------+-----+
|Bronx    |5376 |
|Brooklyn |2688 |
|Manhattan|672  |
|Queens   |384  |
+---------+-----+



## Step 8: Validate Borough — Spatial Recovery + 5% Threshold

In [74]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: Validate borough — spatial recovery + 5% threshold
# ═══════════════════════════════════════════════════════════════════════════════

total = df.count()
null_boro_before = df.filter(F.col("borough").isNull()).count()
print(f"Borough NULL check:")
print(f"   Total rows:   {total:,}")
print(f"   NULL borough: {null_boro_before:,} ({null_boro_before/total*100:.2f}%)")

# 8a: Spatial join recovery
if null_boro_before > 0 and REPROJECTION_DONE and SPATIAL_AVAILABLE:
    print("\nAttempting spatial join recovery...")
    import requests as req

    BORO_GEOJSON_URL = (
        "https://data.cityofnewyork.us/api/geospatial/tqmj-j8zm"
        "?method=export&type=GeoJSON"
    )
    try:
        resp = req.get(BORO_GEOJSON_URL, timeout=30)
        resp.raise_for_status()
        boro_geojson = resp.json()

        boro_polygons = []
        for feature in boro_geojson["features"]:
            boro_name = feature["properties"].get("boro_name", "")
            geom = shape(feature["geometry"])
            boro_polygons.append((boro_name, geom))

        boro_polygons_bc = spark.sparkContext.broadcast(boro_polygons)

        @F.udf(StringType())
        def assign_borough(lat, lon):
            if lat is None or lon is None:
                return None
            try:
                point = Point(lon, lat)
                for boro_name, polygon in boro_polygons_bc.value:
                    if polygon.contains(point):
                        return boro_name
                return None
            except Exception:
                return None

        df = df.withColumn(
            "borough",
            F.when(
                F.col("borough").isNull() & F.col("sensor_lat").isNotNull(),
                assign_borough(F.col("sensor_lat"), F.col("sensor_lon"))
            ).otherwise(F.col("borough"))
        )
        null_after_spatial = df.filter(F.col("borough").isNull()).count()
        print(f"   Recovered {null_boro_before - null_after_spatial:,} boroughs")
    except Exception as e:
        print(f"   Spatial recovery failed: {e}")

elif null_boro_before > 0:
    print("\nSpatial recovery skipped")

# 8b: Flag remaining NULL boroughs for quarantine (don't drop)
THRESHOLD_PCT = 5.0
null_boro_final = df.filter(F.col("borough").isNull()).count()
null_pct = null_boro_final / total * 100 if total > 0 else 0

if null_boro_final == 0:
    print(f"\nStep 8: No NULL boroughs")
elif null_pct < THRESHOLD_PCT:
    print(f"\n   {null_pct:.2f}% NULL boroughs < {THRESHOLD_PCT}% threshold — flagging for quarantine")
    df = df.withColumn(
        "_flag_null_borough_recoverable",
        F.when(F.col("borough").isNull(), "NULL_BOROUGH_BELOW_THRESHOLD").otherwise(None)
    )
    print(f"   {null_boro_final:,} rows flagged for quarantine")
else:
    raise Exception(
        f"PIPELINE ALERT: {null_boro_final:,} rows ({null_pct:.1f}%) have NULL borough. "
        f"Exceeds {THRESHOLD_PCT}% threshold."
    )

print(f"\nStep 8 complete")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 76, Finished, Available, Finished, False)

Borough NULL check:
   Total rows:   9,120
   NULL borough: 0 (0.00%)

Step 8: No NULL boroughs

Step 8 complete


## Step 9: Aggregate 15-Minute to Hourly per Borough

In [75]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONSOLIDATE QUARANTINE FLAGS — Create _quarantine_reasons column
# ══════════════════════════════════════════════════════════════════════════════


flag_cols = [c for c in df.columns if c.startswith("_flag_")]

if flag_cols:
    df = df.withColumn(
        "_quarantine_reasons",
        F.when(
            F.coalesce(*[F.col(c) for c in flag_cols]).isNotNull(),
            F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in flag_cols])
        ).otherwise(None)
    )
else:
    df = df.withColumn("_quarantine_reasons", F.lit(None))

# Drop individual flag columns (keep only _quarantine_reasons)
df = df.drop(*flag_cols)

# Count quarantine candidates
quarantine_candidates = df.filter(F.col("_quarantine_reasons").isNotNull()).count()
clean_candidates = df.filter(F.col("_quarantine_reasons").isNull()).count()

print(f"\n{'='*65}")
print(f"QUARANTINE SUMMARY (before aggregation)")
print(f"{'='*65}")
print(f"   Clean rows (no flags):      {clean_candidates:,}")
print(f"   Quarantine rows (flagged):  {quarantine_candidates:,}")

if quarantine_candidates > 0:
    print(f"\n   Quarantine reasons:")
    df.filter(F.col("_quarantine_reasons").isNotNull()) \
        .groupBy("_quarantine_reasons").count() \
        .orderBy(F.desc("count")).show(truncate=False)

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 77, Finished, Available, Finished, False)


QUARANTINE SUMMARY (before aggregation)
   Clean rows (no flags):      9,120
   Quarantine rows (flagged):  0


In [76]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9: Aggregate 15-min → hourly per borough (clean rows only)
# ═══════════════════════════════════════════════════════════════════════════════

# Split before aggregation
df_quarantine = df.filter(F.col("_quarantine_reasons").isNotNull())
df_clean = df.filter(F.col("_quarantine_reasons").isNull()).drop("_quarantine_reasons")

pre_agg = df_clean.count()

df_hourly = (
    df_clean.withColumn("hour", F.date_trunc("hour", F.col("timestamp")))
      .groupBy("borough", "hour")
      .agg(
          F.sum("vol").alias("traffic_volume"),
          F.count("*").alias("sensor_readings"),
          F.countDistinct("segmentid").alias("sensor_count"),
      )
      .orderBy("borough", "hour")
)

post_agg = df_hourly.count()

print(f"Step 9: Aggregation complete")
print(f"   Before (15-min rows):  {pre_agg:,}")
print(f"   After  (hourly rows):  {post_agg:,}")
if pre_agg > 0:
    print(f"   Compression ratio:     {pre_agg / post_agg:.1f}x")

print(f"\n   Sample hourly data:")
df_hourly.show(10, truncate=False)

print(f"\n   Per-borough summary:")
df_hourly.groupBy("borough").agg(
    F.count("*").alias("hours"),
    F.sum("traffic_volume").alias("total_vehicles"),
    F.avg("traffic_volume").cast("int").alias("avg_hourly_vol"),
    F.avg("sensor_count").cast("int").alias("avg_sensors_per_hour"),
).show(truncate=False)

row_log.append(("After Step 9 (hourly aggregation)", post_agg))

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 78, Finished, Available, Finished, False)

Step 9: Aggregation complete
   Before (15-min rows):  9,120
   After  (hourly rows):  696
   Compression ratio:     13.1x

   Sample hourly data:
+-------+-------------------+--------------+---------------+------------+
|borough|hour               |traffic_volume|sensor_readings|sensor_count|
+-------+-------------------+--------------+---------------+------------+
|Bronx  |2026-01-20 00:00:00|190           |16             |4           |
|Bronx  |2026-01-20 01:00:00|114           |16             |4           |
|Bronx  |2026-01-20 02:00:00|88            |16             |4           |
|Bronx  |2026-01-20 03:00:00|91            |16             |4           |
|Bronx  |2026-01-20 04:00:00|113           |16             |4           |
|Bronx  |2026-01-20 05:00:00|219           |16             |4           |
|Bronx  |2026-01-20 06:00:00|679           |16             |4           |
|Bronx  |2026-01-20 07:00:00|1346          |16             |4           |
|Bronx  |2026-01-20 08:00:00|1258      

In [77]:
print("── Segment coverage diagnostic ──────────────────────────────")
print(f"   Total 15-min rows in Bronze : {pre_agg:,}")
print(f"   Total hourly rows in Silver : {post_agg:,}")
print(f"   Implied avg segments/borough/hour: {pre_agg / post_agg / 4:.1f}")
print()

# How many unique segments exist per borough in the raw data?
df_clean.groupBy("borough").agg(
    F.countDistinct("segmentid").alias("unique_segments"),
    F.count("*").alias("total_readings"),
    F.countDistinct(F.date_trunc("hour", F.col("timestamp"))).alias("hours_covered"),
).orderBy("borough").show(truncate=False)

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 79, Finished, Available, Finished, False)

── Segment coverage diagnostic ──────────────────────────────
   Total 15-min rows in Bronze : 9,120
   Total hourly rows in Silver : 696
   Implied avg segments/borough/hour: 3.3

+---------+---------------+--------------+-------------+
|borough  |unique_segments|total_readings|hours_covered|
+---------+---------------+--------------+-------------+
|Bronx    |4              |5376          |336          |
|Brooklyn |4              |2688          |168          |
|Manhattan|1              |672           |168          |
|Queens   |4              |384           |24           |
+---------+---------------+--------------+-------------+



## Write to Silver Delta Table

In [78]:
# ── Delete last 50 rows from Silver Traffic (incremental write test) ──────────
'''
from delta.tables import DeltaTable

cutoff = (
    spark.table(SILVER_TABLE)
    .orderBy(F.col("hour").desc())
    .limit(50)
    .select("borough", "hour")
)

print("Rows to be deleted:")
cutoff.show(10, truncate=False)

dt = DeltaTable.forName(spark, SILVER_TABLE)
dt.delete(
    F.struct("borough", "hour").isin(
        [F.struct(F.lit(r["borough"]), F.lit(r["hour"])) for r in cutoff.collect()]
    )
)

after_count = spark.table(SILVER_TABLE).count()
print(f" Deleted 50 rows — Silver Traffic now has {after_count:,} rows")
print(f"   Re-run the pipeline and confirm it writes exactly 50 rows back.")
'''

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 80, Finished, Available, Finished, False)

'\nfrom delta.tables import DeltaTable\n\ncutoff = (\n    spark.table(SILVER_TABLE)\n    .orderBy(F.col("hour").desc())\n    .limit(50)\n    .select("borough", "hour")\n)\n\nprint("Rows to be deleted:")\ncutoff.show(10, truncate=False)\n\ndt = DeltaTable.forName(spark, SILVER_TABLE)\ndt.delete(\n    F.struct("borough", "hour").isin(\n        [F.struct(F.lit(r["borough"]), F.lit(r["hour"])) for r in cutoff.collect()]\n    )\n)\n\nafter_count = spark.table(SILVER_TABLE).count()\nprint(f"✅ Deleted 50 rows — Silver Traffic now has {after_count:,} rows")\nprint(f"   Re-run the pipeline and confirm it writes exactly 50 rows back.")\n'

In [79]:
# ═══════════════════════════════════════════════════════════════════════════════
# WRITE TO SILVER + QUARANTINE (incremental — only new rows)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Silver ────────────────────────────────────────────────────────────────────
try:
    spark.table(SILVER_TABLE).limit(1).collect()
    silver_exists = True
except Exception:
    silver_exists = False

if silver_exists:
    existing_keys = spark.table(SILVER_TABLE).select("borough", "hour")
    df_to_write = df_hourly.join(existing_keys, on=["borough", "hour"], how="left_anti")
    already_present = df_hourly.count() - df_to_write.count()
    print(f"Silver table exists — incremental write mode")
    print(f"   Rows in this batch : {df_hourly.count():,}")
    print(f"   Already in Silver  : {already_present:,} (skipped)")
else:
    df_to_write = df_hourly
    print(f"Silver table does not exist — full initial load")

rows_to_write = df_to_write.count()
if rows_to_write > 0:
    df_to_write.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(SILVER_TABLE)
    print(f"\n Written {rows_to_write:,} new rows → {SILVER_TABLE}")
else:
    print(f"\n No new rows to write — Silver table is already up to date")

silver_total = spark.table(SILVER_TABLE).count()
print(f"   Total rows in Silver now: {silver_total:,}")

# ── Quarantine ────────────────────────────────────────────────────────────────
if df_quarantine.count() > 0:
    df_quarantine_final = df_quarantine.withColumn("processed_at", F.current_timestamp())

    try:
        spark.table(QUARANTINE_TABLE).limit(1).collect()
        q_existing = spark.table(QUARANTINE_TABLE).select("borough", F.date_trunc("hour", "timestamp").alias("hour"))
        df_q_to_write = df_quarantine_final.join(q_existing, on=["borough", "hour"], how="left_anti")
    except Exception:
        df_q_to_write = df_quarantine_final

    q_rows = df_q_to_write.count()
    if q_rows > 0:
        df_q_to_write.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(QUARANTINE_TABLE)
        print(f"   Quarantine: {q_rows:,} new rows → {QUARANTINE_TABLE}")
    else:
        print(f"   Quarantine: no new rows")
else:
    print(f"\nNo quarantined rows in this batch")

print(f"\nSilver table schema:")
spark.table(SILVER_TABLE).printSchema()

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 81, Finished, Available, Finished, False)

Silver table exists — incremental write mode
   Rows in this batch : 696
   Already in Silver  : 646 (skipped)

✅ Written 50 new rows → silver_stg_traffic
   Total rows in Silver now: 696

No quarantined rows in this batch

Silver table schema:
root
 |-- borough: string (nullable = true)
 |-- hour: timestamp (nullable = true)
 |-- traffic_volume: long (nullable = true)
 |-- sensor_readings: long (nullable = true)
 |-- sensor_count: long (nullable = true)



## Data Quality Summary

In [80]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA QUALITY SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("DATA QUALITY REPORT -- stg_traffic_v3")
print("=" * 65)

for step_name, count in row_log:
    print(f"   {step_name:50s} {count:>10,}")

print("=" * 65)
print(f"\nstg_traffic_v3 complete -- Silver table ready for Gold join")

print(f"\nFeature status:")
print(f"   Step 3 (coordinate reprojection): {'Done' if REPROJECTION_DONE else 'Skipped'}")
print(f"   Step 4 (fromst/tost recovery):    Dictionary recovery done")
print(f"   Step 5b (segmentid/street):        Dictionary recovery done")
print(f"   Step 8 (spatial boro recovery):    {'Done' if REPROJECTION_DONE else 'Skipped'}")

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 82, Finished, Available, Finished, False)

DATA QUALITY REPORT -- stg_traffic_v3
   Bronze (after dedup + 2026 filter)                      9,120
   After Step 1 (date validation)                          9,120
   After Step 2 (timestamp rebuild)                        9,120
   After Step 5a (requestid + vol)                         9,120
   After Step 9 (hourly aggregation)                         696

stg_traffic_v3 complete -- Silver table ready for Gold join

Feature status:
   Step 3 (coordinate reprojection): Skipped
   Step 4 (fromst/tost recovery):    Dictionary recovery done
   Step 5b (segmentid/street):        Dictionary recovery done
   Step 8 (spatial boro recovery):    Skipped


## Verification Queries

In [81]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFICATION QUERIES
# ── COLAB CHANGE: register temp view for SQL queries ──
# ═══════════════════════════════════════════════════════════════════════════════

silver = spark.table(SILVER_TABLE)
silver.createOrReplaceTempView("silver_stg_traffic")

print("--- Row counts per borough ---")
spark.sql("SELECT borough, COUNT(*) as rows FROM silver_stg_traffic GROUP BY borough ORDER BY borough").show()

print("--- Date range per borough ---")
spark.sql("""
    SELECT borough, MIN(hour) as earliest, MAX(hour) as latest,
           COUNT(DISTINCT DATE(hour)) as days
    FROM silver_stg_traffic GROUP BY borough ORDER BY borough
""").show(truncate=False)

print("--- Any duplicate (borough, hour) pairs? ---")
spark.sql("""
    SELECT borough, hour, COUNT(*) as n
    FROM silver_stg_traffic
    GROUP BY borough, hour HAVING COUNT(*) > 1
""").show()

print("--- Sample rows ---")
silver.show(5, truncate=False)

StatementMeta(, 903866af-7da5-4682-a1f2-11e32e3029b7, 83, Finished, Available, Finished, False)

--- Row counts per borough ---
+---------+----+
|  borough|rows|
+---------+----+
|    Bronx| 336|
| Brooklyn| 168|
|Manhattan| 168|
|   Queens|  24|
+---------+----+

--- Date range per borough ---
+---------+-------------------+-------------------+----+
|borough  |earliest           |latest             |days|
+---------+-------------------+-------------------+----+
|Bronx    |2026-01-20 00:00:00|2026-02-02 23:00:00|14  |
|Brooklyn |2026-01-03 00:00:00|2026-01-09 23:00:00|7   |
|Manhattan|2026-01-10 00:00:00|2026-01-16 23:00:00|7   |
|Queens   |2026-02-03 00:00:00|2026-02-03 23:00:00|1   |
+---------+-------------------+-------------------+----+

--- Any duplicate (borough, hour) pairs? ---
+-------+----+---+
|borough|hour|  n|
+-------+----+---+
+-------+----+---+

--- Sample rows ---
+-------+-------------------+--------------+---------------+------------+
|borough|hour               |traffic_volume|sensor_readings|sensor_count|
+-------+-------------------+--------------+----------